In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from scipy.stats import pearsonr, spearmanr

PROCESSED_DIR = Path("../data/processed")

file_path = (
    PROCESSED_DIR
    / "seoul_apartment_rental_2023_2025_clean.csv"
)

df = pd.read_csv(
    file_path,
    parse_dates=["contract_date"]
)

df.shape

(881794, 25)

In [2]:
df_stat = df[
    (df["contract_date"] >= "2023-01-01")
    & (df["contract_date"] <= "2025-12-31")
].copy()

print(df_stat.shape)

(875925, 25)


In [3]:
df_jeonse = df_stat[
    df_stat["전월세구분"] == "전세"
].copy()

print("전체 분석 데이터:", len(df_stat))
print("전세 데이터:", len(df_jeonse))

전체 분석 데이터: 875925
전세 데이터: 505485


## H1. 자치구별 전세가격 차이

### 가설

서울 아파트의 전세가격은 자치구에 따라 차이가 존재한다.

### 분석 방법

전세가격 분포는 오른쪽 꼬리가 길고 극단값이 존재하므로,
정규분포를 강하게 가정하는 ANOVA보다 비모수 검정인
Kruskal-Wallis 검정을 우선 사용한다.

분석에는 아파트 크기의 영향을 줄이기 위해
단순 보증금이 아닌 단위면적당 보증금(deposit_per_m2)을 사용한다.

귀무가설(H0):
25개 자치구의 단위면적당 전세가격 분포에는 차이가 없다.

대립가설(H1):
적어도 하나 이상의 자치구에서 단위면적당 전세가격 분포가 다르다.

In [4]:
from scipy.stats import kruskal

print("전세 데이터:", len(df_jeonse))
print("자치구 수:", df_jeonse["자치구명"].nunique())

전세 데이터: 505485
자치구 수: 25


In [5]:
gu_groups = [
    group["deposit_per_m2"].dropna().values
    for _, group in df_jeonse.groupby("자치구명")
]

h_stat, p_value = kruskal(*gu_groups)

print("Kruskal-Wallis H:", round(h_stat, 4))
print("p-value:", p_value)

Kruskal-Wallis H: 157921.5082
p-value: 0.0


In [6]:
n = len(df_jeonse)
k = df_jeonse["자치구명"].nunique()

epsilon_squared = (
    (h_stat - k + 1)
    / (n - k)
)

print(
    "Epsilon squared:",
    round(epsilon_squared, 4)
)

Epsilon squared: 0.3124


### H1 결과

Kruskal-Wallis 검정 결과:

- H statistic: 157,921.5082
- p-value: < 0.001
- Epsilon squared: 0.3124

귀무가설은 기각되었다.

서울 25개 자치구의 단위면적당 전세가격 분포에는
통계적으로 유의한 차이가 존재한다.

또한 Epsilon squared가 0.3124로 나타나
자치구에 따른 가격 차이의 효과 크기도 큰 수준으로 확인되었다.

따라서 EDA에서 관찰된 지역별 전세가격 차이는
단순한 표본 변동만으로 설명하기 어려우며,
지역은 전세가격과 강하게 관련된 변수로 판단된다.

단, 본 분석은 관찰 데이터에 기반하므로
자치구 자체가 가격 차이의 직접적인 원인이라고 해석하지 않는다.

## H2. 건축연식과 전세가격

### 가설

건축연식이 낮은 아파트일수록
단위면적당 전세가격이 높은 경향이 존재한다.

### 분석 방법

건축연식과 단위면적당 전세가격 사이의
단조 관계를 확인하기 위해 Spearman 순위상관분석을 사용한다.

귀무가설(H0):
건축연식과 단위면적당 전세가격 사이에 관계가 없다.

대립가설(H1):
건축연식이 증가할수록 단위면적당 전세가격이 낮아지는 경향이 있다.

In [7]:
df_jeonse_age = df_jeonse[
    df_jeonse["has_build_year"]
].copy()

spearman_age_r, spearman_age_p = spearmanr(
    df_jeonse_age["building_age"],
    df_jeonse_age["deposit_per_m2"]
)

print(
    "Spearman rho:",
    round(spearman_age_r, 4)
)

print(
    "p-value:",
    spearman_age_p
)

Spearman rho: -0.4143
p-value: 0.0


### H2 결과

Spearman 순위상관분석 결과:

- Spearman rho: -0.4143
- p-value: < 0.001

건축연식과 단위면적당 전세가격 사이에는
통계적으로 유의한 중간 수준의 음의 상관관계가 나타났다.

즉 건축연식이 증가할수록
단위면적당 전세가격이 낮아지는 경향이 관찰되었다.

이는 EDA에서 확인한 신축 아파트의 높은 전세가격 패턴과
일관된 결과이다.

따라서 H2는 통계분석에서도 지지되는 것으로 판단한다.

다만 건축연식과 지역, 입지, 단지 특성 등이 서로 관련되어 있을 수 있으므로
본 결과만으로 건축연식이 전세가격 하락의 직접적인 원인이라고 해석하지 않는다.

이러한 교란요인은 이후 H4의 다중회귀분석에서 추가로 통제한다.

## H3. 전용면적과 전세가격

### 가설

전용면적이 증가할수록
전세보증금이 증가하는 경향이 존재한다.

### 분석 방법

전용면적과 전세보증금 사이의 관계를 확인하기 위해
Pearson 상관분석과 Spearman 순위상관분석을 함께 수행한다.

- Pearson: 두 변수의 선형 관계 확인
- Spearman: 이상치와 비선형성을 고려한 단조 관계 확인

귀무가설(H0):
전용면적과 전세보증금 사이에 관계가 없다.

대립가설(H1):
전용면적이 증가할수록 전세보증금이 증가하는 경향이 있다.

In [8]:
pearson_area_r, pearson_area_p = pearsonr(
    df_jeonse["임대면적"],
    df_jeonse["보증금(만원)"]
)

spearman_area_r, spearman_area_p = spearmanr(
    df_jeonse["임대면적"],
    df_jeonse["보증금(만원)"]
)

print("Pearson r:", round(pearson_area_r, 4))
print("Pearson p-value:", pearson_area_p)

print()

print("Spearman rho:", round(spearman_area_r, 4))
print("Spearman p-value:", spearman_area_p)

Pearson r: 0.6088
Pearson p-value: 0.0

Spearman rho: 0.6373
Spearman p-value: 0.0


### H3 결과

상관분석 결과:

- Pearson r: 0.6088
- Pearson p-value: < 0.001
- Spearman rho: 0.6373
- Spearman p-value: < 0.001

전용면적과 전세보증금 사이에는
통계적으로 유의한 양의 상관관계가 나타났다.

Pearson 상관계수는 0.6088,
Spearman 순위상관계수는 0.6373으로 확인되어
전용면적이 증가할수록 전세보증금도 증가하는 경향이
비교적 강하게 나타났다.

Spearman 상관계수가 Pearson보다 조금 높게 나타난 점을 고려하면,
면적과 가격 사이의 관계는 명확한 증가 경향을 가지지만
완전히 선형적인 관계만으로 설명되지는 않을 가능성이 있다.

따라서 H3는 통계분석에서도 지지되는 것으로 판단한다.

다만 지역, 건축연식 등 다른 변수들도 전세가격에 영향을 줄 수 있으므로
전용면적의 독립적인 관계는 이후 다중회귀분석에서 추가로 확인한다.

## H4. 다른 조건을 통제한 건축연식과 전세가격의 관계

### 가설

지역, 전용면적, 층, 계약연도 등 다른 조건을 통제한 후에도
건축연식은 전세가격과 유의한 관계를 가진다.

### 분석 방법

전세가격은 오른쪽 꼬리가 긴 분포를 보이므로
보증금에 로그 변환을 적용한 뒤 다중선형회귀분석을 수행한다.

종속변수:
- log(전세보증금)

독립변수:
- 전용면적
- 건축연식
- 층
- 자치구
- 계약연도

자치구와 계약연도는 범주형 변수로 처리한다.

귀무가설(H0):
다른 조건을 통제하면 건축연식과 전세가격 사이의 관계가 없다.

대립가설(H1):
다른 조건을 통제한 후에도 건축연식과 전세가격 사이에 관계가 존재한다.

In [9]:
import statsmodels.formula.api as smf

In [10]:
reg_df = df_jeonse[
    df_jeonse["has_build_year"]
    & (df_jeonse["보증금(만원)"] > 0)
].copy()

reg_df = reg_df[
    [
        "보증금(만원)",
        "임대면적",
        "building_age",
        "층",
        "자치구명",
        "contract_year"
    ]
].dropna()

reg_df["log_deposit"] = np.log(
    reg_df["보증금(만원)"]
)

print("회귀분석 표본:", len(reg_df))

reg_df.head()

회귀분석 표본: 505418


,보증금(만원),임대면적,building_age,층,자치구명,contract_year,log_deposit
0,46000,84.71,20.0,3.0,중랑구,2023,10.736397
1,46500,101.97,20.0,4.0,중랑구,2023,10.747208
2,70000,133.03,19.0,5.0,양천구,2023,11.156251
3,37500,84.94,17.0,14.0,중랑구,2023,10.532096
4,70000,84.98,11.0,6.0,성동구,2023,11.156251


In [11]:
model_h4 = smf.ols(
    formula="""
        log_deposit
        ~ 임대면적
        + building_age
        + 층
        + C(자치구명)
        + C(contract_year)
    """,
    data=reg_df
).fit(
    cov_type="HC1"
)

print(model_h4.summary())

                            OLS Regression Results                            
Dep. Variable:            log_deposit   R-squared:                       0.629
Model:                            OLS   Adj. R-squared:                  0.629
Method:                 Least Squares   F-statistic:                 2.962e+04
Date:                Thu, 08 Oct 2026   Prob (F-statistic):               0.00
Time:                        13:31:02   Log-Likelihood:            -1.9392e+05
No. Observations:              505418   AIC:                         3.879e+05
Df Residuals:                  505388   BIC:                         3.882e+05
Df Model:                          29                                         
Covariance Type:                  HC1                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [12]:
age_coef = model_h4.params["building_age"]
age_p = model_h4.pvalues["building_age"]

print("building_age coefficient:", round(age_coef, 6))
print("building_age p-value:", age_p)
print("R-squared:", round(model_h4.rsquared, 4))
print("Adjusted R-squared:", round(model_h4.rsquared_adj, 4))

building_age coefficient: -0.011911
building_age p-value: 0.0
R-squared: 0.6287
Adjusted R-squared: 0.6287


In [13]:
age_effect_pct = (
    np.exp(age_coef) - 1
) * 100

print(
    "건축연식 1년 증가 시 예상 가격 변화(%):",
    round(age_effect_pct, 3)
)

건축연식 1년 증가 시 예상 가격 변화(%): -1.184


### H4 결과

다중회귀분석 결과:

- building_age coefficient: -0.011911
- p-value: < 0.001
- R-squared: 0.6287
- Adjusted R-squared: 0.6287

지역, 전용면적, 층, 계약연도를 통제한 후에도
건축연식은 전세보증금과 통계적으로 유의한 음의 관계를 보였다.

로그 변환된 전세보증금을 종속변수로 사용했기 때문에
계수를 지수변환하여 해석하면,
건축연식이 1년 증가할 때 전세보증금은 평균적으로
약 1.18% 낮아지는 관계가 나타났다.

즉 동일한 자치구, 면적, 층, 계약연도 조건을 고려하더라도
오래된 아파트일수록 전세가격이 낮은 경향이 확인되었다.

또한 모델의 R-squared는 0.6287로,
포함된 변수들이 전세보증금 변동의 약 62.9%를 설명하는 것으로 나타났다.

따라서 H4는 통계분석에서 지지되는 것으로 판단한다.

다만 본 분석에 포함되지 않은 단지 브랜드, 역세권 여부,
학군, 세대수, 리모델링 여부 등의 요인이 존재할 수 있으므로
건축연식이 가격 차이의 직접적인 원인이라고 해석하지 않는다.

## H5. 보증금과 월세의 관계

### 가설

유사한 지역과 면적 조건에서는
보증금이 증가할수록 월세가 감소하는 경향이 존재한다.

### 배경

EDA에서는 전체 월세 계약을 단순 비교했을 때
보증금과 월세 사이에 약한 양의 관계가 나타났다.

그러나 자치구와 면적을 유사하게 제한했을 때는
모든 자치구에서 음의 상관관계가 관찰되었다.

이는 고가 주택이 보증금과 월세 모두 높은 특성 때문에
전체 데이터에서는 실제 보증금-월세 교환관계가 가려질 가능성을 시사한다.

### 분석 방법

월세와 보증금의 분포가 오른쪽으로 치우쳐 있으므로
두 변수에 로그 변환을 적용한 다중회귀분석을 수행한다.

종속변수:
- log(월세)

주요 독립변수:
- log(보증금)

통제변수:
- 전용면적
- 건축연식
- 층
- 자치구
- 계약연도

귀무가설(H0):
다른 조건을 통제하면 보증금과 월세 사이에 관계가 없다.

대립가설(H1):
다른 조건을 통제하면 보증금이 증가할수록 월세가 감소한다.

In [14]:
df_monthly = df_stat[
    (df_stat["전월세구분"] == "월세")
    & (df_stat["valid_monthly_rent"])
    & (df_stat["임대료(만원)"] > 0)
    & (df_stat["보증금(만원)"] > 0)
    & (df_stat["has_build_year"])
].copy()

monthly_reg = df_monthly[
    [
        "보증금(만원)",
        "임대료(만원)",
        "임대면적",
        "building_age",
        "층",
        "자치구명",
        "contract_year"
    ]
].dropna()

monthly_reg["log_deposit"] = np.log(
    monthly_reg["보증금(만원)"]
)

monthly_reg["log_rent"] = np.log(
    monthly_reg["임대료(만원)"]
)

print("월세 회귀분석 표본:", len(monthly_reg))

월세 회귀분석 표본: 368881


In [15]:
model_h5 = smf.ols(
    formula="""
        log_rent
        ~ log_deposit
        + 임대면적
        + building_age
        + 층
        + C(자치구명)
        + C(contract_year)
    """,
    data=monthly_reg
).fit(
    cov_type="HC1"
)

print(model_h5.summary())

                            OLS Regression Results                            
Dep. Variable:               log_rent   R-squared:                       0.387
Model:                            OLS   Adj. R-squared:                  0.387
Method:                 Least Squares   F-statistic:                     6614.
Date:                Thu, 08 Oct 2026   Prob (F-statistic):               0.00
Time:                        13:32:27   Log-Likelihood:            -4.5449e+05
No. Observations:              368881   AIC:                         9.090e+05
Df Residuals:                  368850   BIC:                         9.094e+05
Df Model:                          30                                         
Covariance Type:                  HC1                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [16]:
deposit_coef = model_h5.params["log_deposit"]
deposit_p = model_h5.pvalues["log_deposit"]

print(
    "log_deposit coefficient:",
    round(deposit_coef, 4)
)

print(
    "log_deposit p-value:",
    deposit_p
)

print(
    "R-squared:",
    round(model_h5.rsquared, 4)
)

print(
    "Adjusted R-squared:",
    round(model_h5.rsquared_adj, 4)
)

log_deposit coefficient: -0.2571
log_deposit p-value: 0.0
R-squared: 0.3871
Adjusted R-squared: 0.387


### H5 결과

다중회귀분석 결과:

- log_deposit coefficient: -0.2571
- p-value: < 0.001
- R-squared: 0.3871
- Adjusted R-squared: 0.3870

자치구, 전용면적, 건축연식, 층, 계약연도를 통제한 후
보증금과 월세 사이에는 통계적으로 유의한 음의 관계가 나타났다.

로그-로그 회귀모형에서 보증금의 계수는 -0.2571로 나타났다.

이는 다른 조건이 동일하다고 가정할 때,
보증금이 1% 증가하면 월세는 평균적으로 약 0.257% 낮아지는
관계가 관찰되었음을 의미한다.

EDA에서 전체 월세 계약을 단순 비교했을 때는
보증금과 월세 사이에 약한 양의 상관관계가 나타났지만,
지역과 주택 특성을 통제한 회귀분석에서는 음의 관계가 확인되었다.

이는 고가 주택의 경우 보증금과 월세가 동시에 높은 특성이 존재하여
전체 데이터의 단순 상관관계가 보증금-월세 간 교환관계를
가릴 수 있음을 보여준다.

따라서 H5는 통계분석에서 지지되는 것으로 판단한다.

다만 본 결과는 관찰 데이터에 기반한 통계적 관계이며,
계약 협상 조건, 임대인의 선호, 시장금리 등
분석에 포함되지 않은 요인이 존재할 수 있으므로
직접적인 인과관계로 해석하지 않는다.

## 가설 검증 결과 요약

EDA에서 발견한 주요 패턴을 통계적으로 검증한 결과,
설정한 다섯 가지 가설 모두 데이터에서 지지되는 결과가 나타났다.

다만 본 분석은 관찰 데이터에 기반하므로
통계적으로 유의한 관계를 직접적인 인과관계로 해석하지 않는다.

In [17]:
hypothesis_results = pd.DataFrame({
    "가설": [
        "H1",
        "H2",
        "H3",
        "H4",
        "H5"
    ],
    "검증 내용": [
        "자치구별 전세가격 차이",
        "건축연식과 단위면적당 전세가격",
        "전용면적과 전세보증금",
        "통제변수 고려 후 건축연식 효과",
        "통제변수 고려 후 보증금-월세 관계"
    ],
    "주요 결과": [
        "ε² = 0.3124",
        "Spearman ρ = -0.4143",
        "Pearson r = 0.6088 / Spearman ρ = 0.6373",
        "연식 1년 증가 시 약 -1.18%",
        "보증금 1% 증가 시 월세 약 -0.257%"
    ],
    "p-value": [
        "< 0.001",
        "< 0.001",
        "< 0.001",
        "< 0.001",
        "< 0.001"
    ],
    "결론": [
        "지지",
        "지지",
        "지지",
        "지지",
        "지지"
    ]
})

hypothesis_results

,가설,검증 내용,주요 결과,p-value,결론
0,H1,자치구별 전세가격 차이,ε² = 0.3124,< 0.001,지지
1,H2,건축연식과 단위면적당 전세가격,Spearman ρ = -0.4143,< 0.001,지지
2,H3,전용면적과 전세보증금,Pearson r = 0.6088 / Spearman ρ = 0.6373,< 0.001,지지
3,H4,통제변수 고려 후 건축연식 효과,연식 1년 증가 시 약 -1.18%,< 0.001,지지
4,H5,통제변수 고려 후 보증금-월세 관계,보증금 1% 증가 시 월세 약 -0.257%,< 0.001,지지


## 통계분석 종합 결론

서울 아파트 전월세 데이터를 대상으로 통계분석을 수행한 결과,
전세가격은 지역, 전용면적, 건축연식과 뚜렷한 관계를 보였다.

특히 자치구별 단위면적당 전세가격 차이는
큰 수준의 효과크기를 보였으며,
전용면적 역시 전세보증금과 비교적 강한 양의 관계를 나타냈다.

건축연식은 지역, 면적, 층, 계약연도를 통제한 이후에도
전세가격과 유의한 음의 관계를 유지했으며,
건축연식이 1년 증가할 때 전세가격은 평균적으로
약 1.18% 낮아지는 관계가 관찰되었다.

월세 시장에서는 전체 데이터를 단순 비교했을 때와 달리
지역과 주택 특성을 통제한 후에는
보증금이 증가할수록 월세가 낮아지는 관계가 확인되었다.

이 결과를 통해 서울 임대시장에서는
지역, 면적, 건축연식과 같은 주택 특성을 동시에 고려하는 것이
가격 구조를 이해하는 데 중요함을 확인하였다.

본 분석은 관찰 데이터에 기반하므로
각 변수와 가격 간의 관계를 직접적인 인과관계로 해석하지 않는다.